# 02 — Feature Analysis

Correlation analysis, feature importance preparation, and feature engineering evaluation.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.features.feature_engineering import engineer_features

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

In [ ]:
df = pd.read_csv('../data/processed/raw_observations.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
print(f"Raw rows: {len(df):,}, columns: {len(df.columns)}")

## 1. Generate Features

In [ ]:
df_feat = engineer_features(df)
print(f"Features: {len(df_feat.columns)} columns")
print(f"Feature columns:")
for i, col in enumerate(sorted(df_feat.columns)):
    print(f"  {i+1:3d}. {col}")

## 2. Correlation Analysis

In [ ]:
# Correlation of features with AQI
numeric_cols = df_feat.select_dtypes(include=[np.number]).columns
target_col = 'aqi'

corrs = {}
for col in numeric_cols:
    if col != target_col and col not in ['hour', 'day_of_week', 'month', 'season']:
        valid = df_feat[[col, target_col]].dropna()
        if len(valid) > 100:
            corrs[col] = valid[col].corr(valid[target_col])

corr_series = pd.Series(corrs).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top_20 = corr_series.head(20)
colors = ['green' if v > 0 else 'red' for v in top_20.values]
ax.barh(range(len(top_20)), top_20.values, color=colors, alpha=0.7)
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20.index)
ax.set_title('Top 20 Features Correlated with AQI')
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Feature Groups Analysis

In [ ]:
# Group features by category
feature_groups = {
    'Time': [c for c in df_feat.columns if c in ['hour', 'day_of_week', 'month', 'season', 'is_weekend', 'hour_sin', 'hour_cos']],
    'Lag': [c for c in df_feat.columns if '_lag_' in c],
    'Rolling': [c for c in df_feat.columns if '_rolling_' in c],
    'Derived': [c for c in df_feat.columns if c in [
        'aqi_change_rate_1h', 'aqi_change_rate_6h', 'aqi_change_rate_24h',
        'aqi_trend_24h', 'pm25_pm10_ratio', 'no2_so2_ratio', 'o3_no2_ratio',
        'temp_humidity_interaction', 'wind_cooling_effect', 'aqi_deviation_from_24h_avg'
    ]],
    'Current': [c for c in df_feat.columns if c in [
        'temperature', 'humidity', 'pressure', 'wind_speed', 'wind_direction',
        'cloud_cover', 'precipitation', 'pm25', 'pm10', 'co', 'no2', 'so2', 'o3'
    ]],
}

print("Feature groups:")
for group, cols in feature_groups.items():
    print(f"  {group:10s}: {len(cols)} features")
    if cols:
        for c in cols[:5]:
            print(f"              {c}")
        if len(cols) > 5:
            print(f"              ... and {len(cols)-5} more")

## 4. Missing Values After Engineering

In [ ]:
# Missing values in engineered features
missing = df_feat.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    print(f"Features with missing values: {len(missing)}")
    fig, ax = plt.subplots(figsize=(12, 6))
    missing.head(20).plot(kind='barh', ax=ax, color='coral')
    ax.set_title('Top 20 Features with Missing Values')
    ax.set_xlabel('Missing Count')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values.')

## 5. Target Analysis

In [ ]:
# Target validity
targets = ['target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']

for t in targets:
    valid = df_feat[t].notna().sum()
    total = len(df_feat)
    print(f"{t}: {valid:,} valid / {total:,} total ({valid/total*100:.1f}%)")

# Target distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, t in zip(axes, targets):
    vals = df_feat[t].dropna()
    ax.hist(vals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{t} (n={len(vals):,})')
    ax.set_xlabel('AQI')
plt.tight_layout()
plt.show()

## 6. Feature Importance Preview (Mutual Information)

In [ ]:
from sklearn.feature_selection import mutual_info_regression

# Use subset for speed
feature_cols = [c for c in df_feat.columns if c not in [
    'timestamp', 'location_id', 'city_name', 'data_source', 'aqi_category',
    'aqi_standard', 'aqi_method', 'aqi_method_version', 'aqi_source',
    'target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h'
] and df_feat[c].dtype in ['float64', 'int64', 'bool']]

# Drop rows with any NaN in features or target
subset = df_feat[feature_cols + ['aqi']].dropna().sample(10000, random_state=42)

X = subset[feature_cols]
y = subset['aqi']

mi = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
mi_series.head(20).plot(kind='barh', ax=ax, color='teal', alpha=0.7)
ax.set_title('Top 20 Features by Mutual Information with AQI')
ax.set_xlabel('MI Score')
plt.tight_layout()
plt.show()